## Pandas with Distributed Files (HDFS)

Lab 01 stored a tiny file and saw it replicated. Here we work with a **larger, real dataset** (MovieLens) and do three things: watch one CSV **fragment into several blocks**, compare its **logical vs physical** size under replication, and run a small **analytics join** reading straight from the Data Lake with Pandas.

#### 1. Connect to HDFS

Check connectivity to the HDFS system and create a client object.

In [ ]:
from hdfs import InsecureClient
try:
    client = InsecureClient('http://localhost:14000', user='root')
    print("Connected!")
except Exception as e:
    print(e)

In [ ]:
# Verify proxy is configured
assert client.list('/') is not None
print('Proxy OK')

#### 2. Check HDFS folders and files

In [ ]:
files = client.list('/')
print(files)

#### 3. Download a public dataset (MovieLens)

[MovieLens](https://grouplens.org/datasets/movielens/) is public — **no authentication**. The small edition is fast; swap the URL for `ml-25m.zip` (~250 MB) for a heavier run. We use two files: `ratings.csv` (~2.4 MB, 100k ratings) and `movies.csv` (titles & genres).

In [ ]:
import urllib.request
import zipfile
import os

os.makedirs('../temp', exist_ok=True)
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
urllib.request.urlretrieve(url, '../temp/movielens.zip')
with zipfile.ZipFile('../temp/movielens.zip', 'r') as zf:
    zf.extractall('../temp/')
os.remove('../temp/movielens.zip')

ratings_csv = '../temp/ml-latest-small/ratings.csv'
movies_csv = '../temp/ml-latest-small/movies.csv'
print(f'ratings.csv: {os.path.getsize(ratings_csv) / 1e6:.2f} MB')
print(f'movies.csv : {os.path.getsize(movies_csv) / 1e6:.2f} MB')

#### 4. Ingest into HDFS

Upload both files through the proxy. By default the ~2.4 MB `ratings.csv` is smaller than the 128 MB block size, so it lands as a **single block**.

In [ ]:
client.makedirs('/movielens', permission=0o755)
client.upload('/movielens/ratings.csv', ratings_csv, overwrite=True)
client.upload('/movielens/movies.csv', movies_csv, overwrite=True)
print('Uploaded:', client.list('/movielens'))

#### 5. Watch one file fragment into many blocks

To *see* fragmentation without huge data, we re-upload `ratings.csv` with a **1 MB block size** (the cluster's minimum). A 2.4 MB file then splits into **3 blocks**, each replicated across the DataNodes — exactly what happens to a multi-GB file at scale, just shrunk down.

In [ ]:
import requests

PROXY = 'http://localhost:14000'


def block_report(hdfs_path):
    """Show an HDFS file's blocks and the DataNodes holding each replica."""
    st = client.status(hdfs_path)
    print(f"{hdfs_path}: size={st['length']:,} | blockSize={st['blockSize']:,} | "
          f"replication={st['replication']}")
    try:
        r = requests.get(f'{PROXY}/webhdfs/v1{hdfs_path}',
                         params={'op': 'GETFILEBLOCKLOCATIONS', 'user.name': 'root'},
                         timeout=30)
        r.raise_for_status()
        blocks = r.json()['BlockLocations']['BlockLocation']
    except Exception as e:
        print(f'  (block locations unavailable via proxy: {e})')
        return
    print(f'  → {len(blocks)} block(s):')
    for i, b in enumerate(blocks):
        print(f"     block {i}: offset={b['offset']:>9,} len={b['length']:>9,} "
              f"on {b['hosts']}")


# Default upload = 1 block
print('=== default block size ===')
block_report('/movielens/ratings.csv')

# Same data, 1 MB blocks = several blocks spread across the cluster
client.upload('/movielens/ratings_small_blocks.csv', ratings_csv,
              overwrite=True, blocksize=1024 * 1024)
print('\n=== 1 MB block size ===')
block_report('/movielens/ratings_small_blocks.csv')

#### 6. Logical vs physical size (the cost of replication)

Replication ×3 means the cluster physically stores **3× the bytes**. The `content` summary makes that visible.

In [ ]:
summary = client.content('/movielens')
logical = summary['length']
physical = summary['spaceConsumed']
print(f"Logical size : {logical / 1e6:.2f} MB ({summary['fileCount']} files)")
print(f"Physical size: {physical / 1e6:.2f} MB on disk across the cluster")
print(f"Replication overhead: {physical / logical:.1f}×")

#### 7. Analytics straight from the Data Lake

Read both files back through the proxy and answer a real question: **which movies are the highest rated** (with enough votes to be meaningful)?

In [ ]:
import pandas as pd
import io

with client.read('/movielens/ratings.csv') as f:
    ratings = pd.read_csv(io.BytesIO(f.read()))
with client.read('/movielens/movies.csv') as f:
    movies = pd.read_csv(io.BytesIO(f.read()))

print(f'{len(ratings):,} ratings, {len(movies):,} movies')

agg = (ratings.groupby('movieId')['rating']
       .agg(['mean', 'count'])
       .query('count >= 50')
       .merge(movies[['movieId', 'title']], on='movieId')
       .sort_values('mean', ascending=False)
       .head(10)[['title', 'mean', 'count']]
       .round({'mean': 2})
       .reset_index(drop=True))
agg

#### 8. Cleanup

In [ ]:
import shutil

client.delete('/movielens', recursive=True)
shutil.rmtree('../temp/ml-latest-small', ignore_errors=True)
print('Cleanup complete ✓')